# Follow-Up Detection

Classify each radiology report for whether it recommends patient-specific follow-up, using an LLM served by Ollama.

**Pipeline:**
1. One-time copy of `default.reports_latest` into a working table `default.reports_followup`. The working table is owned by this notebook so concurrent hl7-transformer ingests don't fight our writes. The schema also carries the demographics, exam, and diagnosis columns that `followup_review_dashboard.py` (the Voilà playbook) reads when building its review UI.
2. Classify reports in batches via Ollama, with a priority filter that processes likely-positive reports (keyword match + complex modality) first.
3. MERGE results back into the working table. Errors go to `default.followup_errors`.

Re-run the top-up cell whenever new HL7 ingests have landed to bring the new accessions into the working table; then re-run the pipeline.

> ℹ️ **Before first run on a policy-enforcing cluster:** this notebook writes via `trino-rw` (the write-enabled Trino instance in `scout-extractor`), which is NetworkPolicy-locked to the `hl7-transformer` pod per ADR 0019. Apply the two policies below to extend access to JupyterHub singleuser pods — one adds ingress on the `trino-rw` side, one adds egress on the Jupyter side (NetworkPolicies are additive). Same shape as the writable-Hive policy admins applied before this notebook moved off Spark.
>
> ```yaml
> # trino-rw-jupyter-access.yaml
> apiVersion: networking.k8s.io/v1
> kind: NetworkPolicy
> metadata:
>   name: trino-rw-jupyter-access
>   namespace: scout-extractor       # trino_rw_namespace
> spec:
>   podSelector:
>     matchLabels:
>       app.kubernetes.io/name: trino
>       app.kubernetes.io/instance: trino-rw
>       app.kubernetes.io/component: coordinator
>   policyTypes:
>     - Ingress
>   ingress:
>     - from:
>         - namespaceSelector:
>             matchLabels:
>               kubernetes.io/metadata.name: scout-analytics   # jupyter_namespace
>           podSelector:
>             matchLabels:
>               app: jupyterhub
>               component: singleuser-server
>       ports:
>         - port: 8080
>           protocol: TCP
> ---
> # jupyter-trino-rw-egress.yaml
> apiVersion: networking.k8s.io/v1
> kind: NetworkPolicy
> metadata:
>   name: jupyter-trino-rw-egress
>   namespace: scout-analytics       # jupyter_namespace
> spec:
>   podSelector:
>     matchLabels:
>       app: jupyterhub
>       component: singleuser-server
>   policyTypes:
>     - Egress
>   egress:
>     - to:
>         - namespaceSelector:
>             matchLabels:
>               kubernetes.io/metadata.name: scout-extractor   # trino_rw_namespace
>           podSelector:
>             matchLabels:
>               app.kubernetes.io/name: trino
>               app.kubernetes.io/instance: trino-rw
>       ports:
>         - port: 8080
>           protocol: TCP
> ```
>
> ```bash
> kubectl apply -f trino-rw-jupyter-access.yaml -f jupyter-trino-rw-egress.yaml
> ```
>
> To revert: `kubectl delete networkpolicy trino-rw-jupyter-access -n scout-extractor; kubectl delete networkpolicy jupyter-trino-rw-egress -n scout-analytics`.

> ⚠️ **Don't "Run All".** Cell 5 is destructive — it drops `default.reports_followup`, wiping any classifier output and reviewer annotations. Run the notebook piecemeal:
> - **First time:** cells 1–3 → 5 (create working table) → 7–9 → 10–11 (test run) → 12–13 (full pipeline)
> - **After new HL7 ingests:** cells 1–3 → **6** (top-up, non-destructive) → 12–13 (classifier resumes from where it stopped)
> - **Anytime:** cell 15 (summary)


In [ ]:
import os
import json
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import trino.dbapi
from tqdm.notebook import tqdm


In [ ]:
# --- Ollama ---
# install-chat (scout-demo) deploys Ollama at ollama.<chatbot_namespace>:11434.
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://ollama.scout-analytics:11434")
MODEL = os.getenv("OLLAMA_MODEL", "gemma4-31b-long:latest")

# Match OLLAMA_NUM_PARALLEL on the deployed Ollama. install-chat doesn't set it,
# so Ollama auto-tunes (~4 on an 80GB A100). For a 31B-long model, 4 is a safe start;
# bump only after benchmarking and confirming GPU memory headroom.
TOTAL_WORKERS = 4
BATCH_SIZE = 1000

# --- Trino RW ---
# trino-rw is the write-enabled Trino instance in scout-extractor (ADR 0019).
# Access requires the NetworkPolicy patches in the header. Auth is anonymous —
# trino-rw is gated by NetworkPolicy, not JWT. The `user` value below shows
# up in trino-rw's audit log for traceability.
TRINO_RW_HOST = os.getenv("TRINO_RW_HOST", "trino-rw.scout-extractor")
TRINO_RW_PORT = int(os.getenv("TRINO_RW_PORT", "8080"))
TRINO_RW_USER = os.getenv("JUPYTERHUB_USER", "followup-notebook")
TRINO_CATALOG = "delta"
TRINO_SCHEMA = "default"

# --- Tables ---
SRC_TABLE = "default.reports_latest"
WORK_TABLE = "default.reports_followup"
ERROR_TABLE = "default.followup_errors"

# --- Priority filter (Trino SQL, used inside WHERE clauses below) ---
FOLLOWUP_KEYWORDS = [
    r"follow.?up", r"repeat", r"recommend", r"suggest", r"interval",
    r"re-?evaluate", r"re-?assess", r"correlat", r"clinical", r"short.?term",
    r"further", r"additional", r"consider", r"refer", r"return",
    r"continue", r"monitor", r"surveillance", r"re-?exam", r"comparison",
]
PRIORITY_MODALITIES = ["CT", "MR", "US", "MG", "PT"]
# Trino: regexp_like(str, pattern). (Spark used the RLIKE infix operator.)
PRIORITY_FILTER = (
    f"regexp_like(lower(report_text), '{'|'.join(FOLLOWUP_KEYWORDS)}') AND "
    f"modality IN ({','.join(repr(m) for m in PRIORITY_MODALITIES)})"
)

print(f"Ollama:   {OLLAMA_URL}  ({MODEL})")
print(f"Workers:  {TOTAL_WORKERS}, batch: {BATCH_SIZE:,}")
print(f"Trino RW: {TRINO_RW_HOST}:{TRINO_RW_PORT} (user={TRINO_RW_USER})")
print(f"Source:   {SRC_TABLE}")
print(f"Work:     {WORK_TABLE}")
print(f"Errors:   {ERROR_TABLE}")


In [ ]:
def _conn():
    """Open a fresh trino-rw connection."""
    return trino.dbapi.connect(
        host=TRINO_RW_HOST,
        port=TRINO_RW_PORT,
        http_scheme="http",
        user=TRINO_RW_USER,
        catalog=TRINO_CATALOG,
        schema=TRINO_SCHEMA,
    )


def trino_query(sql):
    """SELECT against trino-rw; returns a pandas DataFrame."""
    with _conn() as conn:
        cur = conn.cursor()
        cur.execute(sql)
        if cur.description is None:
            return pd.DataFrame()
        cols = [c[0] for c in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)


def trino_execute(sql):
    """DDL/DML against trino-rw; drains the result."""
    with _conn() as conn:
        cur = conn.cursor()
        cur.execute(sql)
        cur.fetchall()


def sql_lit(v):
    """Render a Python value as a SQL literal for inline VALUES clauses.

    Used to build small MERGE / INSERT statements from in-memory pandas
    DataFrames. Booleans, NULLs, and quote-escaped strings are handled;
    anything else is best-effort str()-then-quoted.
    """
    if v is None:
        return "NULL"
    if isinstance(v, bool):
        return "TRUE" if v else "FALSE"
    return "'" + str(v).replace("'", "''") + "'"


# Sanity check — fails fast if the NetworkPolicy isn't applied.
schemas = trino_query("SHOW SCHEMAS")
print(f"trino-rw connected. Schemas: {sorted(schemas[schemas.columns[0]].tolist())}")


## One-time setup: working table

> ⚠️ **Destructive — run only on a fresh deployment.** Drops `default.reports_followup` and recreates it from `reports_latest`. Any prior classifier output and reviewer annotations are lost. After the first run, use the **Top-up** cell below to fold in new accessions without disturbing existing rows.

The fresh schema includes:

- **Key:** `primary_report_identifier` (used for MERGE) and `accession_number` (used by the Voilà playbook).
- **Classifier inputs:** `report_text`, `modality`.
- **Playbook display columns:** `service_name`, `service_identifier`, `message_dt`, `patient_age`, `sex`, `race`, `sending_facility`, `diagnoses`, `principal_result_interpreter`.
- **Classifier outputs:** `followup_detected`, `followup_confidence`, `followup_snippet`, `followup_finding`, `followup_processed_at`.

The cell below tops up new accessions on subsequent runs.


In [ ]:
trino_execute(f"DROP TABLE IF EXISTS {WORK_TABLE}")
print(f"Creating {WORK_TABLE} from {SRC_TABLE} ...")
trino_execute(f"""
    CREATE TABLE {WORK_TABLE} AS
    SELECT
        primary_report_identifier,
        accession_number,
        report_text,
        modality,
        service_name,
        service_identifier,
        message_dt,
        patient_age,
        sex,
        race,
        sending_facility,
        diagnoses,
        principal_result_interpreter,
        CAST(NULL AS BOOLEAN)   AS followup_detected,
        CAST(NULL AS VARCHAR)   AS followup_confidence,
        CAST(NULL AS VARCHAR)   AS followup_snippet,
        CAST(NULL AS VARCHAR)   AS followup_finding,
        CAST(NULL AS TIMESTAMP) AS followup_processed_at,
        -- Reviewer annotation columns, populated by the Voila review
        -- playbook via trino-rw. Pre-created here so the dashboard's
        -- write path doesn't need ALTER TABLE permission.
        CAST(NULL AS BOOLEAN)   AS human_ground_truth,
        CAST(NULL AS VARCHAR)   AS human_notes,
        CAST(NULL AS TIMESTAMP) AS human_reviewed_at
    FROM {SRC_TABLE}
""")
count = int(trino_query(f"SELECT COUNT(*) AS n FROM {WORK_TABLE}")["n"].iloc[0])
print(f"Created. Rows: {count:,}")


In [ ]:
# Top up: insert any rows in SRC_TABLE that aren't yet in WORK_TABLE.
# Run this whenever new HL7 ingests have landed and you want to classify them.
trino_execute(f"""
    INSERT INTO {WORK_TABLE}
    SELECT
        s.primary_report_identifier,
        s.accession_number,
        s.report_text,
        s.modality,
        s.service_name,
        s.service_identifier,
        s.message_dt,
        s.patient_age,
        s.sex,
        s.race,
        s.sending_facility,
        s.diagnoses,
        s.principal_result_interpreter,
        CAST(NULL AS BOOLEAN)   AS followup_detected,
        CAST(NULL AS VARCHAR)   AS followup_confidence,
        CAST(NULL AS VARCHAR)   AS followup_snippet,
        CAST(NULL AS VARCHAR)   AS followup_finding,
        CAST(NULL AS TIMESTAMP) AS followup_processed_at,
        CAST(NULL AS BOOLEAN)   AS human_ground_truth,
        CAST(NULL AS VARCHAR)   AS human_notes,
        CAST(NULL AS TIMESTAMP) AS human_reviewed_at
    FROM {SRC_TABLE} s
    WHERE NOT EXISTS (
        SELECT 1 FROM {WORK_TABLE} w
        WHERE w.primary_report_identifier = s.primary_report_identifier
    )
""")
count = int(trino_query(f"SELECT COUNT(*) AS n FROM {WORK_TABLE}")["n"].iloc[0])
print(f"{WORK_TABLE} rows: {count:,}")


## Classifier

Each classification returns four pieces:
- `follow_up` — whether the report recommends patient-specific follow-up
- `confidence` — `high` or `low`
- `finding` — the underlying clinical finding driving the recommendation (e.g., `"8 mm pulmonary nodule"`). Empty when `follow_up=false`.
- `snippet` — short verbatim excerpt from the report text containing the recommendation. Empty when `follow_up=false`.

`format: "json"` constrains Ollama to emit valid JSON, so we don't need fence-stripping fallbacks. Sampling params live inside `options` — `temperature` at the top level is silently ignored by `/api/generate`. We don't pass `num_ctx`: the model is loaded with the 262144 baked into the `-long` Modelfile, divided across `OLLAMA_NUM_PARALLEL` slots. Overriding per-request would spawn a second runner and double VRAM.


In [ ]:
PROMPT_TEMPLATE = """Does this report recommend follow-up for THIS patient?

Follow-up = patient-specific imaging/evaluation based on findings

YES if:
- Specific directive: "Repeat CT in 6 months for nodule"
- Referral for findings: "Clinical correlation recommended"
- Template applies: Patient has osteoporosis + template says "osteoporosis patients need follow-up"

NO if:
- Normal findings: "normal", "unremarkable", "stable"
- Template doesn't apply: Patient normal + template says "osteoporosis patients need follow-up"
- Generic advice: "all patients should take calcium"
- Report describes a follow-up exam, but does not request any future action

JSON: {{"follow_up": true/false, "confidence": "high"/"low", "finding": "text", "snippet": "text"}}

- finding = the clinical finding requiring follow-up (e.g., "8 mm pulmonary nodule", "indeterminate liver lesion"). "" if follow_up=false.
- snippet = short verbatim excerpt from the report containing the recommendation. "" if follow_up=false.
Read FINDINGS first to check if templates apply.

Report:
{report_text}"""


def classify_report(row):
    try:
        prompt = PROMPT_TEMPLATE.format(report_text=row.report_text)
        resp = requests.post(
            f"{OLLAMA_URL}/api/generate",
            json={
                "model": MODEL,
                "prompt": prompt,
                "stream": False,
                "format": "json",
                "options": {"temperature": 0, "num_predict": 2000},
            },
            timeout=120,
        )
        resp.raise_for_status()
        obj = json.loads(resp.json()["response"])
        confidence = str(obj.get("confidence", "low")).lower()
        if confidence not in ("high", "low"):
            confidence = "low"
        return {
            "primary_report_identifier": row.primary_report_identifier,
            "followup_detected": bool(obj.get("follow_up", False)),
            "followup_confidence": confidence,
            "followup_finding": str(obj.get("finding", "")),
            "followup_snippet": str(obj.get("snippet", "")),
            "error": None,
        }
    except Exception as e:
        return {
            "primary_report_identifier": row.primary_report_identifier,
            "followup_detected": None,
            "followup_confidence": None,
            "followup_finding": None,
            "followup_snippet": None,
            "error": str(e)[:500],
        }


In [ ]:
# Pre-load the model on Ollama and pin it resident (matches OLLAMA_KEEP_ALIVE=-1).
resp = requests.post(
    f"{OLLAMA_URL}/api/generate",
    json={"model": MODEL, "keep_alive": -1},
    timeout=300,
)
resp.raise_for_status()
print(f"{MODEL} resident on Ollama")


## Test run

Classify 100 unprocessed reports end-to-end (without writing back) so you can sanity-check the model's outputs and measure throughput before the full run.


In [ ]:
df_test = trino_query(f"""
    SELECT primary_report_identifier, report_text
    FROM {WORK_TABLE}
    WHERE followup_processed_at IS NULL
      AND ({PRIORITY_FILTER})
    LIMIT 100
""")

if len(df_test) == 0:
    print("No unprocessed priority reports to test on.")
else:
    t0 = time.time()
    results = []
    with ThreadPoolExecutor(max_workers=TOTAL_WORKERS) as ex:
        futures = [ex.submit(classify_report, row) for row in df_test.itertuples(index=False)]
        for f in tqdm(as_completed(futures), total=len(futures), desc="test"):
            results.append(f.result())
    elapsed = time.time() - t0
    errs = sum(1 for r in results if r["error"])
    pos = sum(1 for r in results if r["followup_detected"])
    print(f"\n{len(results)} classified in {elapsed:.1f}s "
          f"({len(results)/elapsed:.2f} reports/sec)")
    print(f"  positives: {pos}, errors: {errs}")
    for r in results[:3]:
        print(f"  - {r}")


## Full pipeline

Loops over batches. Within each batch:
1. Pull `BATCH_SIZE` unprocessed rows (priority-filtered first; falls back to remaining when priority is exhausted).
2. Classify in parallel.
3. Append errors to `ERROR_TABLE`, MERGE successes back into `WORK_TABLE`.

Idempotent — re-running picks up wherever it left off. Stop the cell at any time.


In [ ]:
def fetch_batch(work_table, batch_size, priority_filter):
    """Pull the next batch of unprocessed rows, priority-filtered first."""
    base_where = "followup_processed_at IS NULL"
    df = trino_query(f"""
        SELECT primary_report_identifier, report_text
        FROM {work_table}
        WHERE {base_where} AND ({priority_filter})
        ORDER BY message_dt DESC
        LIMIT {batch_size}
    """)
    if len(df) > 0:
        return df, "priority"
    df = trino_query(f"""
        SELECT primary_report_identifier, report_text
        FROM {work_table}
        WHERE {base_where}
        ORDER BY message_dt DESC
        LIMIT {batch_size}
    """)
    return df, "remaining"


def merge_results(work_table, df_success):
    """MERGE classifier output back into the working table.

    Trino's MERGE on Delta supports the standard SQL syntax (since 418).
    The source is an inline VALUES list — fine at BATCH_SIZE up to a few
    thousand rows; for much larger batches a staging table would scale
    better, but the classifier throughput bounds the batch size well
    below that limit anyway.
    """
    if len(df_success) == 0:
        return
    rows_sql = ",\n            ".join(
        f"({sql_lit(r.primary_report_identifier)}, "
        f"{sql_lit(r.followup_detected)}, "
        f"{sql_lit(r.followup_confidence)}, "
        f"{sql_lit(r.followup_finding)}, "
        f"{sql_lit(r.followup_snippet)})"
        for r in df_success.itertuples(index=False)
    )
    trino_execute(f"""
        MERGE INTO {work_table} AS t
        USING (
            VALUES
            {rows_sql}
        ) AS s(primary_report_identifier, followup_detected, followup_confidence, followup_finding, followup_snippet)
        ON t.primary_report_identifier = s.primary_report_identifier
        WHEN MATCHED THEN UPDATE SET
            followup_detected     = s.followup_detected,
            followup_confidence   = s.followup_confidence,
            followup_finding      = s.followup_finding,
            followup_snippet      = s.followup_snippet,
            followup_processed_at = current_timestamp
    """)


def append_errors(error_table, df_errors):
    """Append classifier failures to the error table; creates it on first call."""
    if len(df_errors) == 0:
        return
    trino_execute(f"""
        CREATE TABLE IF NOT EXISTS {error_table} (
            primary_report_identifier VARCHAR,
            error VARCHAR,
            error_timestamp TIMESTAMP
        )
    """)
    rows_sql = ",\n            ".join(
        f"({sql_lit(r.primary_report_identifier)}, {sql_lit(r.error)}, current_timestamp)"
        for r in df_errors.itertuples(index=False)
    )
    trino_execute(f"""
        INSERT INTO {error_table} (primary_report_identifier, error, error_timestamp)
        VALUES {rows_sql}
    """)


total_unprocessed = int(trino_query(
    f"SELECT COUNT(*) AS n FROM {WORK_TABLE} WHERE followup_processed_at IS NULL"
)["n"].iloc[0])
num_batches = (total_unprocessed + BATCH_SIZE - 1) // BATCH_SIZE
print(f"Unprocessed: {total_unprocessed:,}  |  batches: {num_batches}  |  workers: {TOTAL_WORKERS}")

overall_start = time.time()
total_processed = 0
total_errors = 0

for batch_num in tqdm(range(num_batches), desc="batches"):
    df_batch, source = fetch_batch(WORK_TABLE, BATCH_SIZE, PRIORITY_FILTER)
    if len(df_batch) == 0:
        print("All reports processed.")
        break

    batch_start = time.time()
    results = []
    with ThreadPoolExecutor(max_workers=TOTAL_WORKERS) as ex:
        futures = [ex.submit(classify_report, row) for row in df_batch.itertuples(index=False)]
        with tqdm(total=len(df_batch), desc=f"  batch {batch_num+1} ({source})", leave=False) as pbar:
            for f in as_completed(futures):
                results.append(f.result())
                pbar.update(1)
    batch_elapsed = time.time() - batch_start

    df_results = pd.DataFrame(results)
    df_success = df_results[df_results["error"].isna()].drop(columns=["error"])
    df_err = df_results[df_results["error"].notna()]
    success_count = len(df_success)
    error_count = len(df_err)

    if error_count > 0:
        append_errors(ERROR_TABLE, df_err)
    if success_count > 0:
        merge_results(WORK_TABLE, df_success)

    total_processed += success_count
    total_errors += error_count
    throughput = len(df_batch) / batch_elapsed
    print(f"  batch {batch_num+1}/{num_batches} [{source}]: "
          f"+{success_count} processed, +{error_count} errors, "
          f"{throughput:.2f} reports/sec")

elapsed = time.time() - overall_start
print(f"\nDone. processed={total_processed:,}  errors={total_errors:,}  "
      f"elapsed={elapsed/3600:.2f}h  "
      f"throughput={(total_processed+total_errors)/max(elapsed,1):.2f}/s")


## Summary


In [ ]:
total = int(trino_query(f"SELECT COUNT(*) AS n FROM {WORK_TABLE}")["n"].iloc[0])
processed = int(trino_query(
    f"SELECT COUNT(*) AS n FROM {WORK_TABLE} WHERE followup_processed_at IS NOT NULL"
)["n"].iloc[0])
positives = int(trino_query(
    f"SELECT COUNT(*) AS n FROM {WORK_TABLE} WHERE followup_detected = TRUE"
)["n"].iloc[0])
high_conf_pos = int(trino_query(
    f"SELECT COUNT(*) AS n FROM {WORK_TABLE} "
    f"WHERE followup_detected = TRUE AND followup_confidence = 'high'"
)["n"].iloc[0])

print(f"Total rows:         {total:,}")
print(f"Processed:          {processed:,}  ({100*processed/total:.1f}%)")
print(f"Unprocessed:        {total - processed:,}")
if processed:
    print(f"Follow-up rate:     {100*positives/processed:.1f}%")
if positives:
    print(f"High-confidence:    {high_conf_pos:,}  ({100*high_conf_pos/positives:.1f}% of positives)")

print("\nBy follow-up status / confidence:")
print(trino_query(f"""
    SELECT followup_detected, followup_confidence, COUNT(*) AS count
    FROM {WORK_TABLE}
    GROUP BY followup_detected, followup_confidence
    ORDER BY followup_detected, followup_confidence
""").to_string(index=False))

print("\nSample high-confidence positives:")
print(trino_query(f"""
    SELECT primary_report_identifier, accession_number, modality, followup_finding, followup_snippet
    FROM {WORK_TABLE}
    WHERE followup_detected = TRUE AND followup_confidence = 'high'
    LIMIT 10
""").to_string(index=False))

print("\nMost common findings (high-confidence positives):")
print(trino_query(f"""
    SELECT followup_finding, COUNT(*) AS count
    FROM {WORK_TABLE}
    WHERE followup_detected = TRUE
      AND followup_confidence = 'high'
      AND followup_finding IS NOT NULL
      AND followup_finding != ''
    GROUP BY followup_finding
    ORDER BY count DESC
    LIMIT 15
""").to_string(index=False))

# Error table — might not exist yet on first run.
try:
    err_total = int(trino_query(f"SELECT COUNT(*) AS n FROM {ERROR_TABLE}")["n"].iloc[0])
    print(f"\nErrors logged in {ERROR_TABLE}: {err_total:,}")
    if err_total:
        print(trino_query(f"""
            SELECT error, COUNT(*) AS count
            FROM {ERROR_TABLE}
            GROUP BY error
            ORDER BY count DESC
            LIMIT 5
        """).to_string(index=False))
except Exception:
    pass
